In [1]:
# !pip install langchain langchain-core langchain-community

### Import all necessary libraries

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

### Load Env variables

In [3]:

load_dotenv()

True

### Laod the models

In [4]:
model = ChatOpenAI()

parser = StrOutputParser()

### Simple linear chain

In [5]:
# create prompt template
prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

In [6]:
chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})

print(result)


1. The longest recorded cricket match lasted for 14 days between England and South Africa in 1939, ending in a draw.
2. The highest individual score in a Test cricket match is held by Brian Lara, who scored 400 not out for the West Indies against England in 2004.
3. The game of cricket is believed to have originated in the 16th century in England, making it one of the oldest organized sports in the world.
4. The first international cricket match was played in 1844 between the United States and Canada, making it one of the oldest international sporting rivalries.
5. The Ashes is one of the most famous cricket rivalries between England and Australia, dating back to 1882 when Australia beat England for the first time on English soil, leading to the mocking obituary of English cricket in a newspaper.


In [7]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


### Example 2 - Sequential Chain

In [8]:
prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

In [9]:
chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic':'Data Science'})
print(result)

1. Data science combines statistical analysis, machine learning, computer science, and domain knowledge to extract insights and knowledge from data.
2. Key components of data science include data collection, data cleaning, data exploration, data modeling, and data visualization.
3. Data scientists use programming languages like Python and R, statistical analysis tools like SAS and SPSS, machine learning libraries such as scikit-learn and TensorFlow, as well as data visualization tools like Tableau and Power BI.
4. Data science has diverse applications in industries such as healthcare, finance, retail, marketing, and technology.
5. Data science plays a critical role in enabling businesses to make data-driven decisions and is a promising career field with increasing demand for individuals with strong quantitative and analytical skills.


In [10]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

### Parallel Chains

In [11]:
model2 = ChatOpenAI()

In [12]:
prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

In [13]:
parser = StrOutputParser()
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel({
    'notes': prompt1 | model | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model | parser

chain = parallel_chain | merge_chain



In [14]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

In [15]:
result = chain.invoke({'text':text})

print(result)

Support Vector Machines (SVMs):
- Support vector machines (SVMs) are supervised learning methods used for classification, regression, and outliers detection.
- Advantages:
  - Effective in high dimensional spaces
  - Still effective in cases where the number of dimensions is greater than the number of samples
  - Memory efficient by using a subset of training points in the decision function
  - Versatile with different Kernel functions that can be specified
- Disadvantages:
  - Risk of over-fitting with many features
  - Do not provide direct probability estimates, require expensive five-fold cross-validation for probabilities

Use in Scikit-learn:
- Scikit-learn supports SVMs for both dense (numpy.ndarray) and sparse (scipy.sparse) sample vectors, with optimal performance on C-ordered numpy.ndarray or scipy.sparse.csr_matrix.
  
Quiz:
1. What are support vector machines used for?
Support vector machines are used for classification, regression, and outliers detection.

2. What are the 

In [16]:
chain.get_graph().print_ascii()

            +---------------------------+            
            | Parallel<notes,quiz>Input |            
            +---------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOpenAI |                    | ChatOpenAI |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

### Conditional Chain

In [17]:
prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback}',
    input_variables=['feedback']
    # partial_variables={'format_instruction':parser2.get_format_instructions()}
)

In [18]:
classifier_chain = prompt1 | model | parser


In [19]:
print(classifier_chain.invoke({'feedback': 'This is a beautiful phone'}))


Positive


- We don't have control over the LLM result, base on LLM result next step is decided, To keep output consistent we need to structure the output, 
- For structuring output used - pydanticOutputParser

In [20]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [21]:
class Feedback(BaseModel):
    #Attribute
    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

In [22]:
parser2 = PydanticOutputParser(pydantic_object=Feedback)


In [23]:
prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)


In [24]:
classifier_chain = prompt1 | model | parser2
print(classifier_chain.invoke({'feedback': 'This is a bad phone'}))


sentiment='negative'


In [25]:
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

#if else conditions
''' format =
branch chain= RunnableBranch(
    (condition1,chain1),
    (condition2,chain2),
    default chain
)
'''

' format =\nbranch chain= RunnableBranch(\n    (condition1,chain1),\n    (condition2,chain2),\n    default chain\n)\n'

In [26]:
prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)


In [27]:
branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)


In [28]:
chain = classifier_chain | branch_chain
print(chain.invoke({'feedback': 'This is a beautiful phone'}))


Thank you for your kind words! I'm so glad to hear that you had a positive experience. Let me know if there's anything else I can help with.


In [29]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
